# DINO Detection — Metadata Extraction Pipeline (local PC)

Runs your fine-tuned DINOv2 + Faster R-CNN detector over **all** images in one or more folders
and saves per-image detection metadata (class, confidence, box, ground point) to JSON Lines
files. Built for your PC specifically — no Kaggle paths, no auto-detection assumptions.

## Resumable by design
Metadata is written **incrementally, one line per image**, and already-processed images are
skipped on re-run. If this gets interrupted (closed accidentally, PC sleeps, crashes) partway
through a large folder, just re-run the same cell — it won't reprocess what's already done or
lose what it already saved.

## What this does NOT include (see other notebooks for these)
- No visualization — this is meant to run unattended over a large folder. Use
  `dino_detection_quicktest.ipynb` first to sanity-check the detector on a handful of images.
- No pitch masking / team clustering / coverage — see `pitch_analysis_pipeline.ipynb` for that.
- No bird's-eye view — see the bird's-eye-view notebook if you want that on top of this output.


In [18]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision.ops import MultiScaleRoIAlign, nms as torch_nms
import torchvision.transforms.functional as TF
from PIL import Image
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cpu":
    print("WARNING: no GPU detected. Detection over a large folder will be slow on CPU — "
          "consider running on a smaller subset first, or on a machine with a GPU if this is "
          "meant to process thousands of images.")


Using device: cuda


In [19]:
# ============================================================
# CONFIG — all local paths, no Kaggle auto-detection.
# ============================================================
CONFIG = {
    # Your fine-tuned checkpoint. (On Windows, use forward slashes or an r"..." raw string.)
    "checkpoint_path": r"C:\Users\nishi\code\nishit-notebooks\Data\models\last_checkpoint.pt",   # e.g. "C:/models/best_detector.pt" or "./best_detector.pt"

    # One entry per folder of images you want processed. Add/remove keys as needed —
    # not limited to train/test, name them however makes sense for your data.
    "image_folders": {
        "train": r"C:\Users\nishi\code\nishit-notebooks\Data\Images",   # e.g. "C:/data/Football dataset/train/images"
        "test": None,    # e.g. "C:/data/Football dataset/test/images"
    },

    "metadata_output_dir": "./metadata",   # one <name>_metadata.jsonl file per folder above

    "score_threshold": 0.5,                # default confidence threshold
    "score_threshold_per_class": {
        "player": 0.7,                     # stricter — favor precision for your most numerous class
    },
    "nms_iou_threshold": 0.4,              # lower = more aggressive duplicate-box suppression

    "limit_per_folder": None,              # set an int to cap images processed (testing), or
                                            # None to process ALL images in each folder
}

if not CONFIG["checkpoint_path"]:
    raise RuntimeError("Set CONFIG['checkpoint_path'] to your local fine-tuned checkpoint.")

active_folders = {name: path for name, path in CONFIG["image_folders"].items() if path}
if not active_folders:
    raise RuntimeError("Set at least one path in CONFIG['image_folders'].")

os.makedirs(CONFIG["metadata_output_dir"], exist_ok=True)


In [20]:
# ============================================================
# Load the trained model — architecture is reconstructed from the checkpoint's own saved config,
# so it automatically matches whatever backbone size / freeze mode / img_size you trained with.
# ============================================================
PATCH_SIZE = 14
EMBED_DIMS = {
    "dinov2_vits14": 384, "dinov2_vitb14": 768,
    "dinov2_vitl14": 1024, "dinov2_vitg14": 1536,
}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class DinoV2Backbone(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        # pretrained=False: load_state_dict() below overwrites these weights with your fine-tuned
        # ones anyway, so downloading Meta's generic pretrained weights would be wasted bandwidth.
        self.model = torch.hub.load("facebookresearch/dinov2", model_name, pretrained=False)
        self.embed_dim = EMBED_DIMS[model_name]
        self.out_channels = self.embed_dim
        for p in self.model.parameters():
            p.requires_grad = False
        self.model.eval()

    def forward(self, x):
        b, c, h, w = x.shape
        with torch.no_grad():
            feats = self.model.forward_features(x)
        patch_tokens = feats["x_norm_patchtokens"]
        gh, gw = h // PATCH_SIZE, w // PATCH_SIZE
        feat_map = patch_tokens.permute(0, 2, 1).reshape(b, self.embed_dim, gh, gw)
        return {"0": feat_map}


checkpoint = torch.load(CONFIG["checkpoint_path"], map_location=device)
train_config = checkpoint["config"]
class_names = checkpoint["class_names"]
num_classes = len(class_names)
print(f"Loaded checkpoint trained with model={train_config['model_name']}, "
      f"img_size={train_config['img_size']}, classes={class_names}")

backbone = DinoV2Backbone(train_config["model_name"])
anchor_generator = AnchorGenerator(sizes=((16, 32, 64, 128, 256),), aspect_ratios=((0.5, 1.0, 2.0),))
roi_pooler = MultiScaleRoIAlign(featmap_names=["0"], output_size=7, sampling_ratio=2)
model = FasterRCNN(
    backbone, num_classes=num_classes + 1,
    rpn_anchor_generator=anchor_generator, box_roi_pool=roi_pooler,
    min_size=train_config["img_size"], max_size=train_config["img_size"],
    image_mean=IMAGENET_MEAN, image_std=IMAGENET_STD,
)
model.transform = GeneralizedRCNNTransform(
    min_size=train_config["img_size"], max_size=train_config["img_size"],
    image_mean=IMAGENET_MEAN, image_std=IMAGENET_STD, size_divisible=PATCH_SIZE,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()
print("Model loaded and ready.")


Loaded checkpoint trained with model=dinov2_vits14, img_size=630, classes=['ball', 'goalkeeper', 'player', 'referee']


Using cache found in C:\Users\nishi/.cache\torch\hub\facebookresearch_dinov2_main


Model loaded and ready.


In [21]:
# ============================================================
# Detection — per-class confidence threshold + an extra per-class NMS pass on top of the
# model's own internal NMS, to reduce near-duplicate overlapping boxes on the same object.
# ============================================================
def run_detection(image_pil, score_threshold, score_threshold_per_class, nms_iou_threshold):
    score_threshold_per_class = score_threshold_per_class or {}
    img_tensor = TF.to_tensor(image_pil).to(device)
    with torch.no_grad():
        pred = model([img_tensor])[0]

    boxes, labels, scores = pred["boxes"].cpu(), pred["labels"].cpu(), pred["scores"].cpu()

    keep = torch.tensor([
        scores[i] >= score_threshold_per_class.get(class_names[labels[i].item() - 1], score_threshold)
        for i in range(len(scores))
    ], dtype=torch.bool)
    boxes, labels, scores = boxes[keep], labels[keep], scores[keep]

    final_indices = []
    for cls_id in labels.unique():
        cls_mask = (labels == cls_id).nonzero(as_tuple=True)[0]
        kept = torch_nms(boxes[cls_mask], scores[cls_mask], nms_iou_threshold)
        final_indices.extend(cls_mask[kept].tolist())

    detections = []
    for i in final_indices:
        x1, y1, x2, y2 = boxes[i].tolist()
        detections.append({
            "class": class_names[labels[i].item() - 1],
            "confidence": float(scores[i]),
            "bbox_px": [x1, y1, x2, y2],
            "ground_point_px": [(x1 + x2) / 2, y2],  # bottom-center: where they're standing
        })
    return detections


In [22]:
# ============================================================
# Batch run, per folder — resumable: reads the output file first to see which images are
# already done, and skips them. Writes one JSON object per line as it goes (not all at the end),
# so an interruption never loses already-completed work.
# ============================================================
image_exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def load_already_done(output_path):
    done = set()
    if os.path.exists(output_path):
        with open(output_path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    record = json.loads(line)
                    done.add(record["image_path"])
                except (json.JSONDecodeError, KeyError):
                    continue  # tolerate a corrupted trailing line from a previous interrupted run
    return done


def process_folder(name, images_dir, output_dir, limit=None):
    image_files = sorted(f for f in os.listdir(images_dir) if f.lower().endswith(image_exts))
    if limit is not None:
        image_files = image_files[:limit]

    output_path = os.path.join(output_dir, f"{name}_metadata.jsonl")
    already_done = load_already_done(output_path)
    remaining = [f for f in image_files
                 if os.path.join(images_dir, f) not in already_done]

    print(f"\n--- {name}: {len(image_files)} total, {len(already_done)} already done, "
          f"{len(remaining)} remaining ---")

    if not remaining:
        print("Nothing to do.")
        return

    with open(output_path, "a") as f:
        for fname in tqdm(remaining, desc=f"Processing {name}"):
            image_path = os.path.join(images_dir, fname)
            try:
                image_pil = Image.open(image_path).convert("RGB")
                detections = run_detection(image_pil, CONFIG["score_threshold"],
                                            CONFIG["score_threshold_per_class"],
                                            CONFIG["nms_iou_threshold"])
                record = {"image_path": image_path, "detections": detections}
            except Exception as e:
                # Log and continue rather than losing the whole batch to one bad file.
                record = {"image_path": image_path, "error": str(e)}
                print(f"ERROR on {fname}: {e}")

            f.write(json.dumps(record) + "\n")
            f.flush()  # ensure it's actually on disk immediately, not just buffered

    print(f"Done. Metadata written to {output_path}")


for name, images_dir in active_folders.items():
    process_folder(name, images_dir, CONFIG["metadata_output_dir"], CONFIG["limit_per_folder"])



--- train: 2689 total, 0 already done, 2689 remaining ---


Processing train: 100%|██████████| 2689/2689 [03:55<00:00, 11.44it/s]

Done. Metadata written to ./metadata\train_metadata.jsonl


## Notes

- **Output format**: JSON Lines (`.jsonl`), one JSON object per image, at
  `<metadata_output_dir>/<name>_metadata.jsonl`. Read it back with:
  ```python
  import json
  records = [json.loads(line) for line in open("./metadata/train_metadata.jsonl")]
  ```
- **Resuming**: just re-run the last cell — it checks the output file for already-processed
  `image_path` entries and skips them. Safe to stop and restart as many times as needed.
- **A bad/corrupt image** doesn't stop the whole run — its error gets logged as a record with an
  `"error"` field instead of `"detections"`, and processing continues to the next image.
- **Runtime on CPU**: if you don't have a GPU, expect this to be considerably slower per image
  than on Kaggle's T4 — start with `limit_per_folder` set to something small (e.g. 50) to gauge
  how long a full folder would realistically take before committing to the whole thing.
